In [ ]:
# ==============================================================================
# BƯỚC 3 v14-RandomPruning – Random Token Pruning (Ablation/Baseline)
# Mục đích: ablation study — nếu random pruning tốt gần bằng attention pruning
#           thì attention score không thực sự hữu ích.
#
# Thay vì dùng attention score (eq.2) để chọn token, ta chọn NGẪU NHIÊN
# round(M * ratio) token từ tập content tokens.
#
# Để kết quả ổn định: average qua N_RANDOM_SEEDS lần sample mỗi query/ratio.
# X-axis = Top-k ratio (tỉ lệ token GIỮ LẠI).
# ==============================================================================
print(">>> BƯỚC 3 v14-RandomPruning: Random Token Pruning (Ablation)")

import torch
import torch.nn.functional as F

# ==============================================================================
# CONFIG
# ==============================================================================
IMPORTANCE_VARIANT = 'random_pruning_topk_v14'

# Top-k ratios: tỉ lệ token ĐƯỢC GIỮ LẠI (0.1 = giữ 10%)
TOPK_RATIOS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# Số lần sample ngẫu nhiên để average (giảm variance)
N_RANDOM_SEEDS = 1

# ==============================================================================
# UTILS
# ==============================================================================

def build_content_mask(inputs, processor):
    """Mask: 1 cho content token, 0 cho special/padding."""
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)
    if input_ids is None:
        return attn_mask.float()
    special_ids = set()
    tok = getattr(processor, 'tokenizer', processor)
    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None:
            special_ids.add(int(tid))
    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))
    if not special_ids:
        return attn_mask.float()
    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special     = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()


# ==============================================================================
# PRE-NORMALIZE DOCS
# ==============================================================================
def precompute_docs(docs_list, device):
    n_docs = len(docs_list)
    if n_docs == 0:
        return None, None
    D       = docs_list[0].shape[-1]
    max_len = max(d.shape[0] for d in docs_list)
    doc_pad  = torch.zeros(n_docs, max_len, D, device=device, dtype=torch.float32)
    doc_mask = torch.zeros(n_docs, max_len,    device=device, dtype=torch.bool)
    for i, d in enumerate(docs_list):
        L = d.shape[0]
        doc_pad[i, :L]  = F.normalize(d.float().to(device), dim=-1)
        doc_mask[i, :L] = True
    return doc_pad, doc_mask


# ==============================================================================
# MAXSIM SCORING
# ==============================================================================
def token_doc_maxsim_matrix_precomp(q_norm, doc_pad, doc_mask):
    """
    Args:
        q_norm   : (M, D)
        doc_pad  : (n_docs, max_len, D)
        doc_mask : (n_docs, max_len)
    Returns:
        scores   : (n_docs,)
    """
    sim = torch.einsum('md,nld->mnl', q_norm, doc_pad)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values.sum(dim=0)


# ==============================================================================
# RANDOM PRUNING — ALL TOP-K RATIOS, AVERAGED OVER N_RANDOM_SEEDS
# ==============================================================================
def random_prune_topk(
    q_embed,       # (S, D) full query embedding
    content_mask,  # (S,)  0/1
    doc_pad,       # (n_docs, max_len, D)
    doc_mask,      # (n_docs, max_len)
    topk_ratios,   # list[float]
    n_seeds=N_RANDOM_SEEDS,
):
    """
    Với mỗi ratio r:
      - Tính k = round(M * r)
      - Sample ngẫu nhiên k token từ content tokens, n_seeds lần
      - Average scores qua các seeds → kết quả ổn định hơn 1 lần sample

    Returns:
        dict {ratio: scores (n_docs,)}
    """
    content_idx  = torch.where(content_mask > 0)[0]   # (M,)
    M            = content_idx.numel()
    n_docs       = doc_pad.shape[0]
    device       = q_embed.device

    if M == 0:
        return {r: torch.zeros(n_docs, device=device) for r in topk_ratios}

    content_vecs = F.normalize(q_embed[content_idx].float(), dim=-1)   # (M, D)

    results = {}
    for ratio in topk_ratios:
        k = max(1, round(M * ratio))
        k = min(k, M)

        if k == M:
            # Giữ tất cả → không cần sample
            scores = token_doc_maxsim_matrix_precomp(content_vecs, doc_pad, doc_mask)
            results[ratio] = scores
            continue

        # Average qua n_seeds lần random sample
        accumulated = torch.zeros(n_docs, device=device, dtype=torch.float32)
        for seed in range(n_seeds):
            # torch.randperm đảm bảo không trùng lặp trong 1 sample
            perm       = torch.randperm(M, device=device)[:k]
            pruned     = content_vecs[perm]   # (k, D)
            accumulated += token_doc_maxsim_matrix_precomp(pruned, doc_pad, doc_mask)

        results[ratio] = accumulated / n_seeds

    return results


# ==============================================================================
# EXTRACTION (không cần output_attentions — đơn giản hơn v13)
# ==============================================================================
def extract_embeddings_v14_random(model, inputs, processor):
    try:
        with torch.no_grad():
            outputs = model(**inputs, return_dict=True)
        if hasattr(outputs, "last_hidden_state"):
            proj = outputs.last_hidden_state
        elif isinstance(outputs, torch.Tensor):
            proj = outputs
        else:
            raise RuntimeError("Unknown model output format")
        proj = proj / (proj.norm(dim=-1, keepdim=True) + 1e-8)
        proj = proj * inputs["attention_mask"].unsqueeze(-1).float()
        return proj
    except Exception as e:
        print(f"❌ Extraction error: {e}")
        B, S = inputs['input_ids'].shape
        D    = model.config.hidden_size
        return torch.zeros(B, S, D, device=inputs['input_ids'].device)


print(">>> BƯỚC 3 v14 Random Pruning COMPLETE")


# ==============================================================================
# BƯỚC 3.5: UTILITY FUNCTIONS
# ==============================================================================
print(">>> BƯỚC 3.5: Loading utility functions...")

import numpy as np

def compute_ndcg(ranked_indices, gt_set, k):
    dcg  = sum(1.0 / np.log2(r + 2) for r, idx in enumerate(ranked_indices[:k]) if idx in gt_set)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt_set), k)))
    return dcg / idcg if idcg > 0 else 0.0

def first_hit(top_k, gt_set):
    for r, idx in enumerate(top_k):
        if idx in gt_set:
            return r + 1
    return -1

print("✅ Utility functions loaded")
print(">>> BƯỚC 3.5 COMPLETE")


# ==============================================================================
# BƯỚC 4: COMPREHENSIVE BATCH EVALUATION
# ==============================================================================
print(">>> BƯỚC 4: Process All Batches (Random Pruning — Top-k Ratio)")
print(f"Processing {len(BATCH_RANGES)} document batches: {BATCH_RANGES}")
print(f"Random seeds per query/ratio: {N_RANDOM_SEEDS}")

import json
import os
import pickle
import gc
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm

WORKING_DIR      = "/kaggle/working"
ANNOTATIONS_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
COLSMOL_DIR      = "/kaggle/input/datasets/nguyenducdung1107/colsmol500m-layoutmmdoc/colsmol500m-pkl"

QUERY_BATCH_SIZE = 50

# Method keys: rand_r10, rand_r20, ... rand_r90
METHOD_RAND = [f"rand_r{int(r*100)}" for r in TOPK_RATIOS]

# ==============================================================================
# MASTER STORAGE
# ==============================================================================
all_query_results = []
all_metrics       = {}
all_batch_stats   = []

def _init_metric():
    return {'r1': 0, 'r5': 0, 'r10': 0, 'n1': 0.0, 'n5': 0.0, 'n10': 0.0}

print("\n" + "=" * 80)
print("BATCH LOOP PROCESSING")
print("=" * 80)

for batch_num, (START_IDX, END_IDX) in enumerate(BATCH_RANGES, 1):
    print(f"\n[{batch_num}/{len(BATCH_RANGES)}] Processing Batch [{START_IDX}:{END_IDX}]")

    # =========================================================================
    # LOAD INDEX
    # =========================================================================
    index_path = os.path.join(COLSMOL_DIR, f"{START_IDX}-{END_IDX}.pkl")
    if not os.path.exists(index_path):
        print(f"  ⚠️  Index not found: {index_path}")
        continue

    try:
        with open(index_path, 'rb') as f:
            saved = pickle.load(f)
        if isinstance(saved, dict):
            fused_embeddings = saved.get('embeddings', [])
        else:
            fused_embeddings = saved
        print(f"  ✅ Loaded index: {len(fused_embeddings)} layouts")
    except Exception as e:
        print(f"  ❌ Error loading index: {e}")
        continue

    # =========================================================================
    # LOAD BATCH DATA
    # =========================================================================
    print("  Loading batch data...")
    try:
        batch_doc_names    = intersection_docs[START_IDX:END_IDX]
        batch_target_files = [jsonl_map[d] for d in batch_doc_names]

        batch_df_orig = pd.read_parquet(PARQUET_PATH)
        batch_df_orig['join_doc_name'] = batch_df_orig['doc_name'].str.replace('.pdf', '', regex=False)
        batch_df_orig = batch_df_orig[batch_df_orig['join_doc_name'].isin(batch_doc_names)]

        dfs = []
        for f in tqdm(batch_target_files, desc="    Reading JSONLs", leave=False):
            try:
                temp = pd.read_json(f, lines=True)
                temp['join_doc_name'] = os.path.basename(f).replace('_layout.jsonl', '')
                if 'layout' in temp.columns:
                    temp = temp.rename(columns={'layout': 'layout_id'})
                cols = ['join_doc_name', 'layout_id', 'vlm_text', 'img_enhanced_path']
                if 'text_level' in temp.columns:
                    cols.append('text_level')
                temp = temp[[c for c in cols if c in temp.columns]]
                dfs.append(temp)
            except Exception:
                pass

        if dfs:
            batch_df_enh = pd.concat(dfs, ignore_index=True)
            batch_df_enh = batch_df_enh.rename(columns={
                'vlm_text':   'vlm_text_enhanced',
                'text_level': 'text_level_enhanced'
            })
        else:
            batch_df_enh = pd.DataFrame()

        batch_df_final = pd.merge(
            batch_df_orig, batch_df_enh,
            on=['join_doc_name', 'layout_id'], how='left'
        )
        batch_df_final = batch_df_final.sort_values(by=['join_doc_name', 'page_id', 'layout_id'])

        def identify_header(row):
            if row.get('type') in ['title', 'section_header', 'header']:
                return str(row.get('text', ''))
            if pd.notna(row.get('text_level_enhanced')):
                return str(row.get('text', ''))
            return np.nan

        batch_df_final['temp_header']    = batch_df_final.apply(identify_header, axis=1)
        batch_df_final['current_section'] = (
            batch_df_final.groupby('join_doc_name')['temp_header']
            .ffill().fillna("General Content")
        )

        enh_image_map = {}
        if os.path.exists(ENHANCED_IMG_DIR):
            for fn in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
                enh_image_map[os.path.basename(fn)] = fn

        def get_best_sources(row):
            img_type, img_data = None, None
            if pd.notna(row.get('img_enhanced_path')):
                fname = os.path.basename(str(row['img_enhanced_path']))
                if fname in enh_image_map:
                    img_type, img_data = 'path', enh_image_map[fname]
            if img_data is None and pd.notna(row.get('image_binary')):
                img_type, img_data = 'binary', row['image_binary']
            raw_content = ""
            if pd.notna(row.get('vlm_text_enhanced')) and len(str(row['vlm_text_enhanced'])) > 5:
                raw_content = str(row['vlm_text_enhanced'])
            elif pd.notna(row.get('text')) and len(str(row['text'])) > 5:
                raw_content = str(row['text'])
            elif pd.notna(row.get('ocr_text')) and len(str(row['ocr_text'])) > 5:
                raw_content = str(row['ocr_text'])
            elif pd.notna(row.get('vlm_text')):
                raw_content = str(row['vlm_text'])
            section    = row.get('current_section', '')
            final_text = f"Section: {section} \n Content: {raw_content}"
            if len(final_text) < 10:
                final_text = "Document layout."
            return pd.Series([img_type, img_data, final_text],
                             index=['img_type', 'img_data', 'final_text'])

        processed = batch_df_final.apply(get_best_sources, axis=1)
        batch_sample_layouts_df = (
            pd.concat([batch_df_final, processed], axis=1)
            .dropna(subset=['img_type'])
            .reset_index(drop=True)
        )
        print(f"  ✅ Batch data: {len(batch_sample_layouts_df)} layouts")

    except Exception as e:
        print(f"  ❌ Error loading batch data: {e}")
        import traceback; traceback.print_exc()
        continue

    # =========================================================================
    # BUILD QA PAIRS
    # =========================================================================
    print("  Building QA pairs...")

    def calculate_iou(box1, box2):
        b1 = list(box1) if not isinstance(box1, list) else box1
        b2 = list(box2) if not isinstance(box2, list) else box2
        x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
        x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
        inter  = max(0, x2 - x1) * max(0, y2 - y1)
        union  = ((b1[2]-b1[0])*(b1[3]-b1[1])) + ((b2[2]-b2[0])*(b2[3]-b2[1])) - inter
        return inter / union if union > 0 else 0.0

    batch_qa_pairs   = []
    batch_doc_lookup = {k: v for k, v in batch_sample_layouts_df.groupby('join_doc_name')}
    batch_avail_docs = set(batch_doc_lookup.keys())

    with open(ANNOTATIONS_PATH, 'r') as f:
        for line in f:
            try:
                doc_data = json.loads(line)
            except Exception:
                continue
            target_doc = doc_data['doc_name'].replace('.pdf', '')
            if target_doc not in batch_avail_docs:
                continue
            doc_layouts    = batch_doc_lookup[target_doc]
            domain         = doc_data.get('domain', 'General')
            col_page       = 'page_idx' if 'page_idx' in doc_layouts.columns else 'page_id'
            current_doc_df = doc_layouts.copy()
            current_doc_df['safe_page'] = (
                pd.to_numeric(current_doc_df[col_page], errors='coerce')
                .fillna(-999).astype(int)
            )
            for q_item in doc_data.get('questions', []):
                gt_indices = []
                if 'layout_mapping' in q_item:
                    for target in q_item['layout_mapping']:
                        t_page, t_bbox = target['page'], target['bbox']
                        try:
                            t_page_int = int(t_page)
                            cands = pd.concat([
                                current_doc_df[current_doc_df['safe_page'] == t_page_int],
                                current_doc_df[current_doc_df['safe_page'] == t_page_int - 1],
                            ])
                            for idx, row in cands.iterrows():
                                if calculate_iou(row['bbox'], t_bbox) > 0.5:
                                    gt_indices.append(int(idx))
                        except Exception:
                            continue
                if gt_indices:
                    batch_qa_pairs.append({
                        'question':   q_item['Q'],
                        'gt_indices': list(set(gt_indices)),
                        'doc_name':   target_doc,
                        'domain':     domain,
                    })

    print(f"  ✅ Found {len(batch_qa_pairs)} QA pairs")

    if len(batch_qa_pairs) == 0:
        print("  ⚠️  No QA pairs, skipping")
        all_batch_stats.append({
            'batch':     f"{START_IDX}-{END_IDX}",
            'n_layouts': len(fused_embeddings),
            'n_qa': 0, 'status': 'no_qa'
        })
        continue

    # =========================================================================
    # EVALUATE
    # =========================================================================
    print(f"  Evaluating {len(batch_qa_pairs)} queries...")

    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        n_docs = len(fused_embeddings)
        total  = len(batch_qa_pairs)

        print("  Pre-computing document embeddings...")
        doc_pad, doc_mask = precompute_docs(fused_embeddings, device)
        print(f"  ✅ doc_pad shape: {doc_pad.shape}")

        batch_query_rows = []

        batch_metrics = {'traditional': _init_metric()}
        for r in TOPK_RATIOS:
            batch_metrics[f"rand_r{int(r*100)}"] = _init_metric()

        def hit_metrics(top10, gt):
            h = first_hit(top10, gt)
            return {
                'r1':  int(h != -1 and h <= 1),
                'r5':  int(h != -1 and h <= 5),
                'r10': int(h != -1 and h <= 10),
                'n1':  compute_ndcg(top10, gt, 1),
                'n5':  compute_ndcg(top10, gt, 5),
                'n10': compute_ndcg(top10, gt, 10),
            }

        for query_batch_start in range(0, total, QUERY_BATCH_SIZE):
            query_batch_end   = min(query_batch_start + QUERY_BATCH_SIZE, total)
            query_batch_range = batch_qa_pairs[query_batch_start:query_batch_end]

            all_docs_gpu = [d.float().to(device) for d in fused_embeddings]

            pbar = tqdm(
                enumerate(query_batch_range),
                total=len(query_batch_range),
                desc=f"    Q[{query_batch_start}:{query_batch_end}]",
                leave=False
            )

            for local_q_idx, item in pbar:
                global_q_idx = query_batch_start + local_q_idx
                question     = item['question']
                gt_set       = set(item['gt_indices'])

                # --- Embed query ---
                q_inputs      = processor.process_queries([question]).to(device)
                q_embed_batch = extract_embeddings_v14_random(model, q_inputs, processor)
                q_embed_batch = q_embed_batch.float()
                q_embed       = q_embed_batch[0]   # (S, D)

                content_mask = build_content_mask(q_inputs, processor)[0].float()   # (S,)

                # --- Baseline: full MaxSim ---
                with torch.no_grad():
                    trad_scores = processor.score_multi_vector(
                        q_embed_batch, all_docs_gpu
                    )[0].to(device)

                trad_top10 = torch.topk(trad_scores, k=min(10, n_docs)).indices.cpu().tolist()
                trad_m     = hit_metrics(trad_top10, gt_set)
                for k_, v in trad_m.items():
                    batch_metrics['traditional'][k_] += v

                query_row = {
                    'batch':        f"{START_IDX}-{END_IDX}",
                    'query_id':     global_q_idx,
                    'doc_name':     item['doc_name'],
                    'domain':       item['domain'],
                    'trad_top10':   str(trad_top10),
                    'trad_r@1':     trad_m['r1'],
                    'trad_r@5':     trad_m['r5'],
                    'trad_r@10':    trad_m['r10'],
                    'trad_ndcg@10': round(trad_m['n10'], 4),
                    'n_random_seeds': N_RANDOM_SEEDS,
                }

                # --- Random Pruning — tất cả ratio, averaged over seeds ---
                ratio_scores = random_prune_topk(
                    q_embed      = q_embed,
                    content_mask = content_mask,
                    doc_pad      = doc_pad,
                    doc_mask     = doc_mask,
                    topk_ratios  = TOPK_RATIOS,
                    n_seeds      = N_RANDOM_SEEDS,
                )

                for r in TOPK_RATIOS:
                    key    = f"rand_r{int(r*100)}"
                    scores = ratio_scores[r]
                    top10  = torch.topk(scores, k=min(10, n_docs)).indices.cpu().tolist()
                    m      = hit_metrics(top10, gt_set)
                    for k_, v in m.items():
                        batch_metrics[key][k_] += v
                    query_row.update({
                        f'{key}_top10':   str(top10),
                        f'{key}_r@10':    m['r10'],
                        f'{key}_ndcg@10': round(m['n10'], 4),
                    })

                batch_query_rows.append(query_row)

            del all_docs_gpu
            gc.collect()
            torch.cuda.empty_cache()
            print(f"      ✓ Q[{query_batch_start}:{query_batch_end}] done, memory freed")

        del doc_pad, doc_mask
        gc.collect()
        torch.cuda.empty_cache()

        all_query_results.extend(batch_query_rows)

        for key, mdict in batch_metrics.items():
            if key not in all_metrics:
                all_metrics[key] = _init_metric()
            for k_ in mdict:
                all_metrics[key][k_] += mdict[k_]

        t_r10_pct = batch_metrics['traditional']['r10'] / total * 100 if total > 0 else 0
        t_ndcg10  = batch_metrics['traditional']['n10'] / total       if total > 0 else 0

        print(f"  ✅ Batch done: {len(batch_query_rows)} Q")
        print(f"     Baseline R@10={t_r10_pct:.1f}%  nDCG@10={t_ndcg10:.4f}")

        all_batch_stats.append({
            'batch':          f"{START_IDX}-{END_IDX}",
            'n_layouts':      len(fused_embeddings),
            'n_queries':      total,
            'trad_recall@10': round(t_r10_pct, 2),
            'trad_ndcg@10':   round(t_ndcg10, 4),
            'status':         'ok'
        })

    except Exception as e:
        print(f"  ❌ Evaluation error: {e}")
        import traceback; traceback.print_exc()
        all_batch_stats.append({
            'batch': f"{START_IDX}-{END_IDX}",
            'status': 'error', 'error': str(e)
        })

    # =========================================================================
    # SAVE & CLEANUP
    # =========================================================================
    try:
        if batch_query_rows:
            pd.DataFrame(batch_query_rows).to_csv(
                os.path.join(WORKING_DIR, f"batch_{START_IDX}_{END_IDX}_queries.csv"),
                index=False
            )
        print("  ✅ Batch files saved")
    except Exception as e:
        print(f"  ❌ Save error: {e}")

    del fused_embeddings, batch_sample_layouts_df, batch_qa_pairs
    del batch_df_orig, batch_df_enh, batch_df_final
    gc.collect()
    torch.cuda.empty_cache()
    print("  ✓ Batch memory cleaned\n")


# ==============================================================================
# FINAL: CONSOLIDATE & SUMMARY
# ==============================================================================
print("=" * 80)
print("CONSOLIDATING RESULTS")
print("=" * 80)

try:
    if all_query_results:
        df_all_q = pd.DataFrame(all_query_results)
        master_q = os.path.join(WORKING_DIR, "MASTER_all_batches_queries.csv")
        df_all_q.to_csv(master_q, index=False)
        print(f"✅ {len(df_all_q)} queries → {master_q}")

    print("\n" + "=" * 80)
    print("OVERALL METRICS")
    print("=" * 80)

    total_q = sum(
        row.get('n_queries', 0)
        for row in all_batch_stats if row.get('status') == 'ok'
    )

    print(f"\n{'Method':<18} {'R@1':>7} {'R@5':>7} {'R@10':>7} "
          f"{'nDCG@1':>9} {'nDCG@5':>9} {'nDCG@10':>9}")
    print("-" * 80)

    ordered_keys = ['traditional'] + METHOD_RAND
    for method in ordered_keys:
        if method not in all_metrics:
            continue
        m    = all_metrics[method]
        r1   = m['r1']  / total_q * 100 if total_q else 0
        r5   = m['r5']  / total_q * 100 if total_q else 0
        r10  = m['r10'] / total_q * 100 if total_q else 0
        nd1  = m['n1']  / total_q       if total_q else 0
        nd5  = m['n5']  / total_q       if total_q else 0
        nd10 = m['n10'] / total_q       if total_q else 0
        print(f"{method:<18} {r1:6.2f}%  {r5:6.2f}%  {r10:6.2f}%  "
              f"{nd1:8.4f}  {nd5:8.4f}  {nd10:8.4f}")

    if all_batch_stats:
        pd.DataFrame(all_batch_stats).to_csv(
            os.path.join(WORKING_DIR, "MASTER_batch_stats.csv"), index=False
        )
        print("\nBatch stats:")
        for row in all_batch_stats:
            print(f"  {row}")

except Exception as e:
    print(f"❌ Final aggregation error: {e}")


# ==============================================================================
# BƯỚC 5: VISUALIZE
# ==============================================================================
print("\n>>> BƯỚC 5: Visualize results")

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

    metrics_to_plot = [
        ('ndcg@10',   'n10', 'nDCG@10',   False),
        ('recall@10', 'r10', 'Recall@10',  True),
        ('ndcg@5',    'n5',  'nDCG@5',    False),
        ('recall@1',  'r1',  'Recall@1',   True),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    axes = axes.flatten()

    COLORS = {
        'rand': '#9B59B6',   # ungu — random pruning
        'base': '#888780',   # xám — baseline
    }
    LABELS = {
        'rand': f'Random Pruning (avg {N_RANDOM_SEEDS} seeds)',
        'base': 'Baseline (full ColSMoL)',
    }

    for ax_idx, (_, mkey, title, pct_flag) in enumerate(metrics_to_plot):
        ax = axes[ax_idx]
        ax.set_title(title, fontsize=12, fontweight='normal', pad=8)
        ax.set_xlabel('Top-k ratio (fraction of tokens kept)', fontsize=10)
        ax.set_ylabel('Recall (%)' if pct_flag else 'Score', fontsize=10)
        ax.grid(axis='y', alpha=0.25, linewidth=0.6)
        ax.grid(axis='x', alpha=0.15, linewidth=0.4)
        ax.spines[['top', 'right']].set_visible(False)

        # Baseline dashed line
        if total_q > 0:
            base_val = all_metrics['traditional'][mkey] / total_q
            if pct_flag: base_val *= 100
            ax.axhline(base_val, color=COLORS['base'], linewidth=1.8,
                       linestyle='--', label=LABELS['base'], zorder=2)

        # Random Pruning line
        vals = []
        for r in TOPK_RATIOS:
            key = f"rand_r{int(r*100)}"
            if key in all_metrics and total_q > 0:
                v = all_metrics[key][mkey] / total_q
                if pct_flag: v *= 100
                vals.append(v)
            else:
                vals.append(None)

        valid = [(r, v) for r, v in zip(TOPK_RATIOS, vals) if v is not None]
        if valid:
            xs, ys = zip(*valid)
            ax.plot(xs, ys, color=COLORS['rand'], linewidth=2.2,
                    marker='o', markersize=5, label=LABELS['rand'], zorder=3)

        ax.set_xticks(TOPK_RATIOS)
        ax.set_xticklabels([str(r) for r in TOPK_RATIOS], fontsize=8)
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda v, _: f"{v:.1f}%" if pct_flag else f"{v:.3f}")
        )
        ax.legend(fontsize=8, framealpha=0.4, loc='lower right')

    plt.suptitle(
        f"Random Token Pruning vs Baseline  (seeds={N_RANDOM_SEEDS})\n"
        "Randomly sample round(M × top-k ratio) tokens, average scores over seeds",
        fontsize=11, y=1.02
    )
    plt.tight_layout()

    plot_path = os.path.join(WORKING_DIR, "random_pruning_topk_comparison.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Plot saved → {plot_path}")

except Exception as e:
    print(f"⚠️  Plot error (non-fatal): {e}")

print("\n>>> BƯỚC 4+5 COMPLETE — ALL BATCHES PROCESSED (Random Pruning — Top-k Ratio)")